# 🎯 Phase 2: Validated PEAR Loop and Advanced Agent Tests

**Status:** ✅ All Tests Passing (4/4)

**Date Validated:** November 17, 2025

These tests validate the PEAR loop (Plan-Execute-Analyze-Review-Replan) and advanced agent interactions.

---

## Test Suite Coverage

| Test ID | Test Name | Status | Feature Tested |
|---------|-----------|--------|----------------|
| 2.1 | Review Agent - needs_data | ✅ PASS | PEAR Loop (Full Replan) |
| 2.2 | Review Agent - needs_refinement | ✅ PASS | PEAR Loop (Partial Retry) |
| 1.5 | Multiple Tool Selection | ✅ PASS | Multi-tool bind_tools |
| 1.6 | Ambiguous Query Handling | ✅ PASS | Scope management |

## 📦 Setup

In [ ]:
import requests
import json
from datetime import datetime
from IPython.display import display, Markdown

BASE_URL = "http://localhost:8000"

def run_test(test_id, test_name, query, cif_no="C000001", expected_behavior=None):
    """Run a single test and display results."""
    print(f"\n{'='*80}")
    print(f"🧪 Test {test_id}: {test_name}")
    print(f"{'='*80}")
    print(f"Query: {query}")
    if expected_behavior:
        print(f"Expected: {expected_behavior}")
    print("\nExecuting (may take 30-60 seconds for complex queries)...")
    
    payload = {
        "query": query,
        "context": {"cif_no": cif_no},
        "user_id": "test_analyst",
        "session_id": f"test_{test_id}_{datetime.now().strftime('%H%M%S')}"
    }
    
    try:
        # Phase 2 tests need longer timeout due to PEAR loop iterations
        response = requests.post(f"{BASE_URL}/api/query", json=payload, timeout=180)
        
        if response.status_code == 200:
            data = response.json()
            print(f"\n✅ Status: {response.status_code} OK")
            print(f"\n📝 Response Preview:")
            print(data['response'][:500] + "..." if len(data['response']) > 500 else data['response'])
            
            # Show compliance analysis if present
            if data.get('compliance_analysis'):
                ca = data['compliance_analysis']
                print(f"\n📊 Compliance Analysis:")
                if ca.get('risk_assessment'):
                    print(f"  Risk Assessment: {ca['risk_assessment']}")
                if ca.get('typologies'):
                    print(f"  Typologies: {ca['typologies']}")
                if ca.get('recommendations'):
                    print(f"  Recommendations: {len(ca['recommendations'])} items")
            
            # Show retrieved data if present
            if data.get('retrieved_data'):
                print(f"\n💾 Data Retrieved: {len(data['retrieved_data'])} fields")
            
            print(f"\n✅ TEST {test_id} PASSED")
            return True, data
        else:
            print(f"\n❌ Status: {response.status_code}")
            print(f"Error: {response.text}")
            print(f"\n❌ TEST {test_id} FAILED")
            return False, None
            
    except requests.Timeout:
        print(f"\n❌ Request timed out (>180s)")
        print(f"\n❌ TEST {test_id} FAILED")
        return False, None
    except Exception as e:
        print(f"\n❌ Exception: {e}")
        print(f"\n❌ TEST {test_id} FAILED")
        return False, None

print("✅ Setup complete!")
print(f"API Base URL: {BASE_URL}")

## 🏥 Pre-Test: Health Check

Verify API is running and healthy before tests.

In [ ]:
response = requests.get(f"{BASE_URL}/health")
health = response.json()

print("🏥 API Health Check")
print("=" * 50)
print(f"Status: {health['status']}")
print(f"Database: {health['database']}")
print(f"Redis: {health['redis']}")
print(f"Agents: {health['agents']}")

if health['status'] == 'healthy':
    print("\n✅ All systems operational! Ready to run tests.")
else:
    print("\n⚠️ Some components unhealthy. Tests may fail.")

---

# Phase 2 Tests

---

## ✅ Test 2.1: Review Agent - needs_data (Full Replan)

**Objective:** Trigger Review Agent to request additional data through PEAR loop

**Expected Flow:** 
```
Coordinator → Intent Mapper → Data Retrieval → Compliance Expert → Review Agent
   ↓ (review_status = needs_data)
Intent Mapper → Data Retrieval → Compliance Expert → Review Agent → END
```

**Validates:**
- ✓ Review Agent identifies when more data is needed
- ✓ PEAR loop routes back to Intent Mapper for additional data
- ✓ Comprehensive analysis generated with all necessary data
- ✓ Multiple tool calls orchestrated correctly

In [ ]:
passed, result = run_test(
    test_id="2.1",
    test_name="Review Agent - needs_data (Full Replan)",
    query="Provide a comprehensive AML risk analysis for this customer",
    cif_no="C000001",
    expected_behavior="Comprehensive analysis with multiple data sources"
)

# Check if comprehensive data was gathered (evidence of PEAR loop)
if result and result.get('retrieved_data'):
    fields = len(result['retrieved_data'])
    if fields >= 5:
        print(f"\n✓ Comprehensive data gathered: {fields} field groups (evidence of PEAR loop)")
    else:
        print(f"\n⚠ Limited data: {fields} field groups")

## ✅ Test 2.2: Review Agent - needs_refinement (Partial Retry)

**Objective:** Trigger Review Agent to request better analysis from Compliance Expert

**Expected Flow:**
```
Coordinator → Intent Mapper → Data Retrieval → Compliance Expert → Review Agent
   ↓ (review_status = needs_refinement)
Compliance Expert → Review Agent → END
```

**Validates:**
- ✓ Review Agent can request refinement without new data
- ✓ Compliance Expert provides more detailed analysis on retry
- ✓ PEAR loop works for iterative improvement
- ✓ Quality control prevents generic responses

In [ ]:
passed, result = run_test(
    test_id="2.2",
    test_name="Review Agent - needs_refinement (Partial Retry)",
    query="What specific AML red flags should I look for with this customer?",
    cif_no="C000001",
    expected_behavior="Specific, detailed red flag guidance"
)

# Check for specific red flag analysis
if result:
    resp_lower = result['response'].lower()
    red_flag_terms = [
        'red flag', 'warning sign', 'indicator', 'suspicious', 'pattern',
        'unusual', 'structuring', 'cash', 'high-risk'
    ]
    found_terms = [term for term in red_flag_terms if term in resp_lower]
    
    if len(found_terms) >= 5:
        print(f"\n✓ Specific red flag analysis: {len(found_terms)} key terms found")
        print(f"  Terms: {', '.join(found_terms[:5])}")
    else:
        print(f"\n⚠ Limited specificity: {len(found_terms)} key terms found")

## ✅ Test 1.5: Intent Mapper - Multiple Tool Selection

**Objective:** Verify Intent Mapper can select and execute multiple tools via bind_tools

**Expected Flow:**
```
Coordinator → Intent Mapper (selects multiple tools) → Data Retrieval (executes all) → 
Compliance Expert → Review Agent → END
```

**Validates:**
- ✓ Intent Mapper identifies multiple data needs from single query
- ✓ bind_tools allows multiple tool selection
- ✓ All selected tools execute successfully
- ✓ Data from multiple tools integrated coherently

In [ ]:
passed, result = run_test(
    test_id="1.5",
    test_name="Intent Mapper - Multiple Tool Selection",
    query="Show me the customer's basic information, their transaction history, and risk assessment",
    cif_no="C000001",
    expected_behavior="Multiple data categories retrieved and integrated"
)

# Check for multiple data categories
if result and result.get('retrieved_data'):
    retrieved = result['retrieved_data']
    data_categories = []
    
    # Check for basic info
    if any(k in str(retrieved) for k in ['name', 'account', 'occupation']):
        data_categories.append('Basic Info')
    
    # Check for transactions
    if any(k in str(retrieved) for k in ['transaction', 'txn', 'amount']):
        data_categories.append('Transactions')
    
    # Check for risk
    if any(k in str(retrieved) for k in ['risk', 'score']):
        data_categories.append('Risk Assessment')
    
    if len(data_categories) >= 2:
        print(f"\n✓ Multiple data categories: {', '.join(data_categories)}")
    else:
        print(f"\n⚠ Limited categories: {', '.join(data_categories) if data_categories else 'None'}")

## ✅ Test 1.6: Ambiguous Query Handling

**Objective:** Test handling of vague queries lacking specificity

**Expected Flow:**
```
Coordinator (identifies vague/out-of-scope query) → END with guidance
```

**Validates:**
- ✓ Coordinator identifies vague queries
- ✓ Provides helpful guidance on AML scope
- ✓ Polite, professional rejection
- ✓ Prevents wasted agent cycles on unclear requests

In [ ]:
passed, result = run_test(
    test_id="1.6",
    test_name="Ambiguous Query Handling",
    query="Tell me about the customer",
    cif_no="C000001",
    expected_behavior="Clarification request OR scope guidance"
)

# Check response type
if result:
    resp_lower = result['response'].lower()
    
    # Check for clarification language
    clarification_terms = ['clarify', 'specific', 'which', 'more information', 'could you']
    has_clarification = any(term in resp_lower for term in clarification_terms)
    
    # Check for scope guidance
    scope_terms = ['aml', 'compliance', 'scope', 'focus', 'assist']
    has_scope = any(term in resp_lower for term in scope_terms)
    
    if has_clarification:
        print("\n✓ Response requests clarification")
    elif has_scope:
        print("\n✓ Response provides scope guidance")
    else:
        print("\n⚠ Response type unclear")

---

## 🔍 Additional Exploration

### Try PEAR Loop Triggering

Test queries designed to trigger specific PEAR loop behaviors:

In [ ]:
# Example: Query likely to trigger needs_data
query = "Analyze this customer for money laundering risk"
cif = "C000002"

print("Testing PEAR loop with comprehensive analysis query...")
passed, result = run_test(
    test_id="CUSTOM-PEAR",
    test_name="Custom PEAR Loop Test",
    query=query,
    cif_no=cif
)

### Try Your Own Query

Customize and test your own queries:

In [ ]:
# Customize these values:
custom_query = "What are the main AML risks for this customer?"
custom_cif = "C000003"

passed, result = run_test(
    test_id="CUSTOM",
    test_name="Custom Query",
    query=custom_query,
    cif_no=custom_cif
)